# Summoning daemons

The daemon runs services side by side until one ends or it is stopped, when the
others are stopped too. A service is anything with `run` and `stop`.

In [1]:
import asyncio

# A service is one long-running job; the daemon runs them together.
from amoorion.daemon import Daemon, Service


# Ticks some times, or endlessly with None, and remembers being stopped.
class Ticker(Service):
    name = "ticker"

    def __init__(
        self,
        times,
    ):

        super().__init__()

        self.times, self.ticks, self.stopped = times, 0, False

    async def run(self):

        while not self.stopped and (
            self.times is None or self.ticks < self.times
        ):
            self.ticks += 1

            await asyncio.sleep(0.01)

    async def stop(self):

        self.stopped = True


# One that ends by itself, one that never would.
short, endless = Ticker(3), Ticker(None)


# The daemon ends when the first service ends, and stops the others.
async def summon():

    await Daemon(
        [
            short,
            endless,
        ]
    ).run()


# A notebook can `await` right away, as it already runs an event loop.
await summon()

# The short one finished, the endless one was stopped.
short.ticks, endless.stopped

(3, True)

A service that fails brings the daemon down with its error, so a supervisor sees it:

In [2]:
# A service that fails as soon as it starts.
class Broken(Service):
    name = "broken"

    async def run(self):

        raise RuntimeError("no connection")


# Its failure comes out of `run`, after the others were stopped.
async def summon_broken():

    try:
        await Daemon(
            [
                Broken(),
                Ticker(None),
            ]
        ).run()
    except RuntimeError as error:
        return f"daemon stopped: {error}"


await summon_broken()

'daemon stopped: no connection'

## Threads and processes

The daemon itself is asyncio: services wait side by side. What blocks goes to a thread, and what fills a processor goes to a process; the service stays the same from the outside.


In [3]:
from concurrent.futures import ProcessPoolExecutor

import time


# Waiting on something blocking: a thread.
class Blocking(Service):
    name = "blocking"

    async def run(self):

        await asyncio.to_thread(
            time.sleep,
            0.01,
        )

        self.answer = "slept in a thread"

    async def stop(self):

        pass


# Heavy computing: a process, as threads would share one processor.
class Heavy(Service):
    name = "heavy"

    async def run(self):

        with ProcessPoolExecutor(1) as pool:
            digits = await asyncio.get_running_loop().run_in_executor(
                pool,
                pow,
                2,
                100000,
            )

        self.answer = f"computed {digits.bit_length()} bits in a process"

    async def stop(self):

        pass


services = [
    Blocking(),
    Heavy(),
]

await Daemon(services).run()

[service.answer for service in services]

['slept in a thread', 'computed 100001 bits in a process']

For real, the API and hooks, and NATS when `.env` names a server:

```sh
uv run amoorion daemon --host 127.0.0.1 --port 8000
```